# predicting future kaggle top performers

This notebook contains my solution for the NEOAI 2026 Kaggleforces competition.

Task: rank users by their probability of finishing in the top 3 percent  
Metric: ROC AUC  
Model: HistGradientBoosting with time-aware user profiles

### a small clarification

The competition provided the task, data, metric, statistical context, and a starter baseline. I wrote and adapted the Kaggle code myself, including the time-aware feature engineering, leakage control, validation, model training, and submission pipeline.

I am currently studying statistics independently, so the terminology here reflects what the problem requires, not a claim that I already know everything.


## approach

Raw rank is not enough cuz competitions have different numbers of teams. Rank 100 can slay in one competition and flop in another. I first convert public and private leaderboard positions into relative ranks.

I then build a user profile from competitions completed before the row being predicted. The features describe experience, submission activity, previous finishes, medals, recent form, score gaps, and rewards.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

In [ ]:
BASE_DIR = Path('/kaggle/input/competitions/neoai-2026-day-1-kaggleforces')
train = pd.read_parquet(BASE_DIR / 'train.parquet')
test = pd.read_csv(BASE_DIR / 'test.csv')
sample = pd.read_csv(BASE_DIR / 'sample_submission.csv')
print(train.shape, test.shape)
train.head()

## data preparation

Dates and numeric columns are converted to consistent types. Missing medal, reward, and submission values are filled with zero. The target is 1 when the private relative rank is at most 0.03.


In [ ]:
numeric_cols = ['TotalTeams', 'PublicLeaderboardRank', 'PrivateLeaderboardRank', 'final_public_score', 'final_private_score', 'n_submissions', 'Medal', 'RewardQuantity']
for col in numeric_cols:
    train[col] = pd.to_numeric(train[col], errors='coerce')
train['EnabledDate'] = pd.to_datetime(train['EnabledDate'], errors='coerce')
train['DeadlineDate'] = pd.to_datetime(train['DeadlineDate'], errors='coerce')
train['TotalTeams'] = train['TotalTeams'].clip(lower=1)
train['Medal'] = train['Medal'].fillna(0)
train['RewardQuantity'] = train['RewardQuantity'].fillna(0)
train['n_submissions'] = train['n_submissions'].fillna(0)
train['public_rel'] = train['PublicLeaderboardRank'] / train['TotalTeams']
train['private_rel'] = train['PrivateLeaderboardRank'] / train['TotalTeams']
train['rank_shake'] = train['private_rel'] - train['public_rel']
train['score_gap'] = (train['final_private_score'] - train['final_public_score']).abs()
train['target'] = (train['private_rel'] <= 0.03).astype('int8')
train = train.sort_values(['DeadlineDate', 'UserId']).reset_index(drop=True)
train['target'].value_counts(normalize=True)

## historical features

Expanding statistics are shifted by one competition, so each training row uses only earlier results. Recent-form features are calculated from the previous three appearances.


In [ ]:
def expanding_mean(frame, col):
    return frame.groupby('UserId', sort=False)[col].transform(lambda s: s.shift().expanding().mean())

def expanding_min(frame, col):
    return frame.groupby('UserId', sort=False)[col].transform(lambda s: s.shift().expanding().min())

def expanding_max(frame, col):
    return frame.groupby('UserId', sort=False)[col].transform(lambda s: s.shift().expanding().max())

def recent_mean(frame, col, window=3):
    return frame.groupby('UserId', sort=False)[col].transform(lambda s: s.shift().rolling(window, min_periods=1).mean())

train['history_count'] = train.groupby('UserId', sort=False).cumcount()
train['avg_private_rel'] = expanding_mean(train, 'private_rel')
train['best_private_rel'] = expanding_min(train, 'private_rel')
train['avg_public_rel'] = expanding_mean(train, 'public_rel')
train['avg_submissions'] = expanding_mean(train, 'n_submissions')
train['max_submissions'] = expanding_max(train, 'n_submissions')
train['top3_rate'] = expanding_mean(train, 'target')
train['medal_rate'] = expanding_mean(train.assign(has_medal=(train['Medal'] > 0).astype(int)), 'has_medal')
train['gold_rate'] = expanding_mean(train.assign(has_gold=(train['Medal'] == 1).astype(int)), 'has_gold')
train['avg_rank_shake'] = expanding_mean(train, 'rank_shake')
train['avg_score_gap'] = expanding_mean(train, 'score_gap')
train['max_reward'] = expanding_max(train, 'RewardQuantity')
train['recent_private_rel'] = recent_mean(train, 'private_rel')
train['recent_top3_rate'] = recent_mean(train, 'target')

## test profiles

For test users, I aggregate all available historical competitions and merge the resulting profiles onto each competition-user pair.


In [ ]:
history = train.sort_values(['UserId', 'DeadlineDate'])
profiles = history.groupby('UserId').agg(
    history_count=('CompetitionId', 'nunique'),
    avg_private_rel=('private_rel', 'mean'),
    best_private_rel=('private_rel', 'min'),
    avg_public_rel=('public_rel', 'mean'),
    avg_submissions=('n_submissions', 'mean'),
    max_submissions=('n_submissions', 'max'),
    top3_rate=('target', 'mean'),
    avg_rank_shake=('rank_shake', 'mean'),
    avg_score_gap=('score_gap', 'mean'),
    max_reward=('RewardQuantity', 'max')
).reset_index()
medal_profiles = history.assign(
    has_medal=(history['Medal'] > 0).astype(int),
    has_gold=(history['Medal'] == 1).astype(int)
).groupby('UserId').agg(
    medal_rate=('has_medal', 'mean'),
    gold_rate=('has_gold', 'mean')
).reset_index()
recent_profiles = history.groupby('UserId').tail(3).groupby('UserId').agg(
    recent_private_rel=('private_rel', 'mean'),
    recent_top3_rate=('target', 'mean')
).reset_index()
profiles = profiles.merge(medal_profiles, on='UserId', how='left').merge(recent_profiles, on='UserId', how='left')
test_features = test.merge(profiles, on='UserId', how='left')

## time-based validation

A random split would mix older and newer competitions. I split by competition deadline instead, using older rows for training and newer rows for validation.


In [ ]:
features = [
    'history_count', 'avg_private_rel', 'best_private_rel', 'avg_public_rel',
    'avg_submissions', 'max_submissions', 'top3_rate', 'medal_rate',
    'gold_rate', 'avg_rank_shake', 'avg_score_gap', 'max_reward',
    'recent_private_rel', 'recent_top3_rate'
]
usable = train[train['history_count'] >= 3].copy()
cutoff = usable['DeadlineDate'].quantile(0.8)
fit_mask = usable['DeadlineDate'] < cutoff
val_mask = usable['DeadlineDate'] >= cutoff
X_fit = usable.loc[fit_mask, features].replace([np.inf, -np.inf], np.nan)
y_fit = usable.loc[fit_mask, 'target']
X_val = usable.loc[val_mask, features].replace([np.inf, -np.inf], np.nan)
y_val = usable.loc[val_mask, 'target']
model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=350,
    max_leaf_nodes=15,
    min_samples_leaf=25,
    l2_regularization=2.0,
    random_state=42
)
model.fit(X_fit, y_fit)
val_pred = model.predict_proba(X_val)[:, 1]
print(f'Time-based ROC-AUC: {roc_auc_score(y_val, val_pred):.5f}')

## final training and submission

The model is retrained on all usable historical rows. It predicts continuous probabilities for the test set, which are saved to `submission.csv`.


In [ ]:
X_all = usable[features].replace([np.inf, -np.inf], np.nan)
y_all = usable['target']
X_test = test_features[features].replace([np.inf, -np.inf], np.nan)
model.fit(X_all, y_all)
predictions = model.predict_proba(X_test)[:, 1]
submission = pd.DataFrame({
    'Id': test['CompetitionId'].astype(str) + '_' + test['UserId'].astype(str),
    'pred_score': predictions
})
submission.to_csv('submission.csv', index=False)
submission.head()

## notes

Relative rank is more informative than raw rank, recent performance matters, and leakage control is essential. Possible next steps include competition-difficulty features, exponentially weighted statistics, model blending, and out-of-fold encodings.
